# run_move_detection: evaluate move detection on recorded games (M5, optional)

Needs recorded games (`data/games/<game_id>/{game.pgn, frames/*.png}` with warped frames,
or a video to warp first) and a trained M3 model. Not executed in CI.
Writes `results/<run_name>/move_metrics.json`.

**TODO**
- [ ] `MoveTracker.process_frame` and `evaluate_on_game` in `move_detection.ipynb`
- [ ] tune the detector: plot frame_difference over time (cell 2)
- [ ] ablation: argmax-FEN diff vs legality-filtered infer_move

In [ ]:
%run ./train.ipynb
%run ./capture.ipynb
%run ./move_detection.ipynb
%cd ..
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

cfg = load_config("configs/train.yaml")
cam_cfg = load_config("configs/camera.yaml")
geom = load_board_geometry()
model = CNNClassifier.from_checkpoint("models/resnet18/mixed/best.pt", cfg)
GAMES = sorted(p for p in Path("data/games").iterdir() if p.is_dir())


def board_proba(warped_bgr):
    crops = [cv2.cvtColor(c, cv2.COLOR_BGR2RGB) for c in extract_all_crops(warped_bgr, geom, cam_cfg.warp, cam_cfg.crop)]
    return model.predict_proba(crops)

## Detector tuning: frame differences over time

In [ ]:
frames = [cv2.imread(str(p)) for p in sorted((GAMES[0] / "frames").glob("*.png"))]
diffs = [frame_difference(a, b) for a, b in zip(frames, frames[1:])]
plt.plot(diffs); plt.axhline(StableFrameDetector().diff_threshold, color="r"); plt.ylabel("mean abs diff")

## Evaluate every game

In [ ]:
results = {g.name: evaluate_on_game(g / "game.pgn", g / "frames", board_proba=board_proba) for g in GAMES}
out = Path("results", cfg.eval.run_name, "move_metrics.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(results, indent=2, default=str))
results